
# Practice Quiz — Pandas Essentials (Solved, Open‑Note)

Purpose: show *what and why* at each step. Each task includes:
1) Short explanation of the Pandas pattern.
2) Idiomatic one‑liner or compact solution.
3) Quick sanity check output.

Datasets:
- `main.csv` — `id, group, city, value, score, status`
- `lookup.csv` — `city → region` mapping


In [ ]:

import pandas as pd

main_path = "/mnt/data/main.csv"
lookup_path = "/mnt/data/lookup.csv"

df = pd.read_csv(main_path)
lookup = pd.read_csv(lookup_path)

df.head(10)



## 1) Replace three values in one column using a single `.replace`

**Pattern:** Map old→new in a dict and pass it to `.replace` on the Series.

- Why: `.replace` handles multiple value substitutions cleanly in one call.
- Gotcha: `.map` would create NaN for unmapped keys; `.replace` keeps non‑targets unchanged.


In [ ]:

# Replace 'old_red','old_green','old_blue' → 'red','green','blue' in one call.
df_q1 = df.copy()
df_q1["status"] = df_q1["status"].replace({"old_red": "red", "old_green": "green", "old_blue": "blue"})

# Sanity check: counts should now show only 'red','green','blue' and no 'old_*'
df_q1["status"].value_counts()



## 2) Filter → group → aggregate → filter, **one line with chaining**

**Pattern:** Use `.loc[lambda d: ...]` before and after `.groupby(...).agg(...)` to keep it piped.

Steps:
1. Keep rows where `score >= 50`.
2. Group by `group`, compute `value_mean` and `score_sum`.
3. Keep only groups with `score_sum >= 150`.

**Why `lambda` in `.loc`?** It preserves chaining without breaking the pipe and avoids intermediate named variables.


In [ ]:

# One chain. No intermediate variables.
res_q2 = (
    df.loc[lambda d: d["score"] >= 50]
      .groupby("group", as_index=False)
      .agg(value_mean=("value","mean"), score_sum=("score","sum"))
      .loc[lambda g: g["score_sum"] >= 150]
)
res_q2



## 3) Recode a numeric variable into categorical intervals

**Pattern:** `pd.cut` with bins and labels.

We will create `value_band` from `value` using half‑open intervals `[0,40)`, `[40,70)`, `[70,∞)`.
- Use `right=False` to make the intervals left‑closed and right‑open.
- Labels produce a tidy string category column.


In [ ]:

bins = [0, 40, 70, float("inf")]
labels = ["low", "mid", "high"]

df_q3 = df.copy()
df_q3["value_band"] = pd.cut(df_q3["value"], bins=bins, labels=labels, right=False)

# Sanity check: display pairs
df_q3[["value","value_band"]].head(12)



## 4) Group → aggregate → sort by an aggregated statistic, **strictly one line using chaining**

**Task:** Group by `city`. Aggregate `n = count of id`, `score_mean = mean score`. Sort by `score_mean` in descending order.

**Constraint:** Put the entire solution on one line. This demonstrates tight chaining literacy.


In [ ]:

# One physically single line. No line breaks.
res_q4 = df.groupby("city", as_index=False).agg(n=("id","count"), score_mean=("score","mean")).sort_values("score_mean", ascending=False); res_q4



## 5) Rename exactly one column while leaving others unchanged

**Pattern:** `DataFrame.rename(columns={old:new})` returns a copy unless `inplace=True`.


In [ ]:

df_q5 = df.rename(columns={"value": "value_points"})
df_q5.head(8)



## 6) Merge two DataFrames to add a lookup column

**Pattern:** Left join to add attributes from a lookup table.

- `df.merge(lookup, on="city", how="left")` ensures all main rows remain, `region` is appended where matched.
- Good habit: choose `how="left"` for enriching the primary table.


In [ ]:

df_q6 = df.merge(lookup, on="city", how="left")
df_q6.head(10)



## Mini cheatsheet

- Multiple replacements in one shot  
  `df["col"] = df["col"].replace({old1:new1, old2:new2, old3:new3})`

- Filter → group → agg → filter, piped  
  `(df.loc[lambda d: cond(d)].groupby(keys, as_index=False).agg(...).loc[lambda g: cond2(g)])`

- Numeric → categorical intervals  
  `df["cat"] = pd.cut(df["num"], bins=[...], labels=[...], right=False)`

- One‑line group→agg→sort  
  `df.groupby(key, as_index=False).agg(...).sort_values(col, ascending=False)`

- Rename single column  
  `df = df.rename(columns={"old":"new"})`

- Left merge lookup  
  `df = df.merge(lookup, on="key", how="left")`
